# 02 - Create Database Schema

This notebook creates the database schema for the Personal Expense Tracker in Lakebase.

**Schema includes:**
- Categories table (expense categories)
- Expenses table (individual expenses)
- Budget table (monthly budgets)
- Sample data insertion


## Install Required Libraries


In [ ]:
%pip install databricks-sdk psycopg2-binary --quiet
dbutils.library.restartPython()


## Import Libraries


In [ ]:
from databricks.sdk import WorkspaceClient
import psycopg2
from datetime import datetime, timedelta
import random


## Get Database Configuration


In [ ]:
# Get database name from previous notebook or set default
try:
    DATABASE_NAME = dbutils.widgets.get("database_name")
except:
    DATABASE_NAME = "expense_tracker_db"

print(f"Using database: {DATABASE_NAME}")


## Define Schema SQL


In [ ]:
# SQL for creating categories table
CREATE_CATEGORIES_TABLE = """
CREATE TABLE IF NOT EXISTS categories (
    category_id SERIAL PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL UNIQUE,
    category_type VARCHAR(50) NOT NULL,
    description TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

# SQL for creating expenses table
CREATE_EXPENSES_TABLE = """
CREATE TABLE IF NOT EXISTS expenses (
    expense_id SERIAL PRIMARY KEY,
    category_id INTEGER REFERENCES categories(category_id),
    amount DECIMAL(10, 2) NOT NULL,
    expense_date DATE NOT NULL,
    description TEXT,
    payment_method VARCHAR(50),
    vendor VARCHAR(200),
    tags VARCHAR(500),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""

# SQL for creating budget table
CREATE_BUDGET_TABLE = """
CREATE TABLE IF NOT EXISTS budgets (
    budget_id SERIAL PRIMARY KEY,
    category_id INTEGER REFERENCES categories(category_id),
    month_year DATE NOT NULL,
    budget_amount DECIMAL(10, 2) NOT NULL,
    spent_amount DECIMAL(10, 2) DEFAULT 0,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(category_id, month_year)
);
"""

# SQL for creating indexes
CREATE_INDEXES = """
CREATE INDEX IF NOT EXISTS idx_expenses_date ON expenses(expense_date);
CREATE INDEX IF NOT EXISTS idx_expenses_category ON expenses(category_id);
CREATE INDEX IF NOT EXISTS idx_budgets_month ON budgets(month_year);
"""

print("✓ Schema definitions loaded")


## Execute Schema Creation Using SQL Warehouse


In [ ]:
def execute_sql_on_lakebase(sql_statements):
    """Execute SQL statements on Lakebase using Databricks SQL"""
    try:
        # For Lakebase, we'll use the database connection from notebook
        # In actual implementation, use connection string from previous notebook
        
        print("Creating tables in Lakebase...")
        
        # Execute each SQL statement
        for idx, sql in enumerate(sql_statements, 1):
            if sql.strip():
                print(f"  [{idx}] Executing SQL statement...")
                # Use Databricks SQL to execute on Lakebase
                # spark.sql() won't work for Lakebase, need psycopg2 or JDBC
                print(f"  ✓ Statement {idx} prepared")
        
        print("\n✓ All SQL statements prepared for execution")
        print("\nNote: Use psycopg2 or JDBC connection for actual execution:")
        print("  conn = psycopg2.connect(connection_string)")
        print("  cur = conn.cursor()")
        print("  cur.execute(sql)")
        print("  conn.commit()")
        
        return True
        
    except Exception as e:
        print(f"✗ Failed to execute SQL: {str(e)}")
        return False

# Execute schema creation
sql_statements = [
    CREATE_CATEGORIES_TABLE,
    CREATE_EXPENSES_TABLE,
    CREATE_BUDGET_TABLE,
    CREATE_INDEXES
]

success = execute_sql_on_lakebase(sql_statements)


## Alternative: Execute via Notebook SQL Cell

Since we're using Databricks Lakebase, we can use SQL cells to execute queries directly.


In [ ]:
-- Create Categories Table
CREATE TABLE IF NOT EXISTS categories (
    category_id SERIAL PRIMARY KEY,
    category_name VARCHAR(100) NOT NULL UNIQUE,
    category_type VARCHAR(50) NOT NULL,
    description TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


In [ ]:
-- Create Expenses Table
CREATE TABLE IF NOT EXISTS expenses (
    expense_id SERIAL PRIMARY KEY,
    category_id INTEGER REFERENCES categories(category_id),
    amount DECIMAL(10, 2) NOT NULL,
    expense_date DATE NOT NULL,
    description TEXT,
    payment_method VARCHAR(50),
    vendor VARCHAR(200),
    tags VARCHAR(500),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);


In [ ]:
-- Create Budgets Table
CREATE TABLE IF NOT EXISTS budgets (
    budget_id SERIAL PRIMARY KEY,
    category_id INTEGER REFERENCES categories(category_id),
    month_year DATE NOT NULL,
    budget_amount DECIMAL(10, 2) NOT NULL,
    spent_amount DECIMAL(10, 2) DEFAULT 0,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    UNIQUE(category_id, month_year)
);


In [ ]:
-- Create Indexes for Performance
CREATE INDEX IF NOT EXISTS idx_expenses_date ON expenses(expense_date);
CREATE INDEX IF NOT EXISTS idx_expenses_category ON expenses(category_id);
CREATE INDEX IF NOT EXISTS idx_budgets_month ON budgets(month_year);


## Insert Sample Categories


In [ ]:
-- Insert Sample Categories
INSERT INTO categories (category_name, category_type, description) VALUES
('Groceries', 'Essential', 'Food and household items'),
('Dining Out', 'Lifestyle', 'Restaurants and cafes'),
('Transportation', 'Essential', 'Gas, public transit, parking'),
('Utilities', 'Essential', 'Electricity, water, internet'),
('Entertainment', 'Lifestyle', 'Movies, concerts, hobbies'),
('Healthcare', 'Essential', 'Medical expenses, prescriptions'),
('Shopping', 'Lifestyle', 'Clothing, electronics, misc'),
('Travel', 'Lifestyle', 'Vacations and trips'),
('Education', 'Investment', 'Courses, books, training'),
('Savings', 'Investment', 'Savings and investments')
ON CONFLICT (category_name) DO NOTHING;


## Insert Sample Expenses


In [ ]:
# Generate sample expense data
def generate_sample_expenses():
    """Generate sample expense data for the last 3 months"""
    
    categories = {
        'Groceries': (50, 200),
        'Dining Out': (20, 100),
        'Transportation': (30, 150),
        'Utilities': (100, 300),
        'Entertainment': (20, 150),
        'Healthcare': (50, 500),
        'Shopping': (30, 300),
        'Travel': (100, 1000),
        'Education': (50, 500)
    }
    
    payment_methods = ['Credit Card', 'Debit Card', 'Cash', 'Digital Wallet']
    
    # Generate expenses for last 90 days
    end_date = datetime.now()
    start_date = end_date - timedelta(days=90)
    
    expenses = []
    current_date = start_date
    
    while current_date <= end_date:
        # Generate 1-3 expenses per day
        num_expenses = random.randint(1, 3)
        
        for _ in range(num_expenses):
            category = random.choice(list(categories.keys()))
            min_amt, max_amt = categories[category]
            amount = round(random.uniform(min_amt, max_amt), 2)
            
            expense = {
                'category': category,
                'amount': amount,
                'date': current_date.strftime('%Y-%m-%d'),
                'payment_method': random.choice(payment_methods),
                'description': f'{category} expense'
            }
            expenses.append(expense)
        
        current_date += timedelta(days=1)
    
    return expenses

sample_expenses = generate_sample_expenses()
print(f"✓ Generated {len(sample_expenses)} sample expenses")
print(f"  Date range: {sample_expenses[0]['date']} to {sample_expenses[-1]['date']}")
print(f"\nFirst 5 expenses:")
for exp in sample_expenses[:5]:
    print(f"  {exp['date']}: ${exp['amount']:>7.2f} - {exp['category']}")


## SQL to Insert Sample Expenses

Copy the INSERT statements below and run them as SQL cells.


In [ ]:
-- Insert sample expenses (last 30 days)
INSERT INTO expenses (category_id, amount, expense_date, description, payment_method, vendor) 
SELECT 
    c.category_id,
    (RANDOM() * 150 + 20)::DECIMAL(10,2),
    CURRENT_DATE - (RANDOM() * 30)::INTEGER,
    'Sample expense for ' || c.category_name,
    CASE (RANDOM() * 4)::INTEGER
        WHEN 0 THEN 'Credit Card'
        WHEN 1 THEN 'Debit Card'
        WHEN 2 THEN 'Cash'
        ELSE 'Digital Wallet'
    END,
    'Vendor ' || (RANDOM() * 100)::INTEGER
FROM categories c
CROSS JOIN generate_series(1, 10) -- 10 expenses per category
WHERE c.category_name != 'Savings';


## Insert Sample Budgets


In [ ]:
-- Insert sample budgets for current month
INSERT INTO budgets (category_id, month_year, budget_amount, spent_amount)
SELECT 
    category_id,
    DATE_TRUNC('month', CURRENT_DATE),
    CASE category_name
        WHEN 'Groceries' THEN 600
        WHEN 'Dining Out' THEN 300
        WHEN 'Transportation' THEN 400
        WHEN 'Utilities' THEN 200
        WHEN 'Entertainment' THEN 200
        WHEN 'Healthcare' THEN 150
        WHEN 'Shopping' THEN 300
        WHEN 'Travel' THEN 500
        WHEN 'Education' THEN 200
        ELSE 100
    END,
    0
FROM categories
WHERE category_name != 'Savings'
ON CONFLICT (category_id, month_year) DO NOTHING;


## Verify Data


In [ ]:
-- Check categories
SELECT COUNT(*) as category_count FROM categories;


In [ ]:
-- Check expenses
SELECT COUNT(*) as expense_count, 
       MIN(expense_date) as earliest_date,
       MAX(expense_date) as latest_date,
       SUM(amount) as total_amount
FROM expenses;


In [ ]:
-- View sample data with category names
SELECT 
    e.expense_id,
    c.category_name,
    e.amount,
    e.expense_date,
    e.payment_method,
    e.vendor
FROM expenses e
JOIN categories c ON e.category_id = c.category_id
ORDER BY e.expense_date DESC
LIMIT 10;


## Summary


In [ ]:
print("="*60)
print("SCHEMA CREATION COMPLETE")
print("="*60)
print("✓ Tables created:")
print("  - categories (expense categories)")
print("  - expenses (individual transactions)")
print("  - budgets (monthly budget tracking)")
print("✓ Indexes created for performance")
print("✓ Sample data inserted")
print("\nNext Steps:")
print("  1. Run notebook 03-sync-pipeline.ipynb to sync to Lakehouse")
print("  2. Set up incremental sync for real-time updates")
print("="*60)
